# BUILD STEP — gather info about the account that posted each tweet.

**Data used:** extracted features (variables we generated with Gemini / Apify / XPOZ)

**Short answer:** No results here; this just makes the data that notebook 12 analyses.



# Build User Feature Dataset

Produces a single CSV (`outputs/poster_features/user_features.csv`) with one row per unique tweet author (890 users), combining three feature layers:

| Layer | Source | Features |
|---|---|---|
| 1 — Simple | `Existing_Users.xlsx` | followers, following, ff_ratio, account_age, has_bio, has_location; raw bio text + avatar URL kept as columns |
| 2 — LLM | Gemini bio scoring | bio stance explicitness (1–5), bio account type |
| 3 — LLM vision | Gemini avatar vision | avatar type, has national/political symbol |
| 4 — XPOZ SDK | XPOZ API | verified, inauthentic_prob_score, inauthentic_type, avg_tweets_per_day |

All API calls are cached — safe to interrupt and resume.

---
## 0. Configuration

In [ ]:
import os

BASE_DIR = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(BASE_DIR, "data")) and os.path.dirname(BASE_DIR) != BASE_DIR:
    BASE_DIR = os.path.dirname(BASE_DIR)
DATA_DIR   = os.path.join(BASE_DIR, "data")
OUT_DIR    = os.path.join(BASE_DIR, "outputs", "poster_features")
AVATAR_DIR = os.path.join(OUT_DIR, "avatar_cache")
os.makedirs(OUT_DIR,    exist_ok=True)
os.makedirs(AVATAR_DIR, exist_ok=True)

# ── API keys ───────────────────────────────────────────────────────────────────
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "YOUR_GEMINI_API_KEY")
GEMINI_MODEL   = "gemini-2.5-flash"
XPOZ_API_KEY   = "YOUR_XPOZ_API_KEY"

# ── Cache file paths ───────────────────────────────────────────────────────────
XPOZ_CACHE        = os.path.join(OUT_DIR, "xpoz_cache.json")
BIO_CACHE         = os.path.join(OUT_DIR, "bio_scoring_cache.json")
AVATAR_SCORE_CACHE = os.path.join(OUT_DIR, "avatar_scoring_cache.json")

API_DELAY = 0.3   # seconds between API calls

print("Output dir:", OUT_DIR)
print("Gemini key set:", GEMINI_API_KEY == "YOUR_GEMINI_API_KEY")

---
## 1. Imports

In [ ]:

import json, time, re, hashlib, warnings
import requests
import numpy as np
import pandas as pd
from google import genai
from google.genai import types
# !pip install xpoz -q
from xpoz import XpozClient   # pip install xpoz  (run once in terminal if missing)

warnings.filterwarnings("ignore")
print("Imports OK")


---
## 2. Load & Filter Users (Layer 1 — Simple Features)

`Existing_Users.xlsx` contains all users ever collected by the pipeline (~22k rows). We filter to only the 890 users who authored tweets in Control or Treatment, then compute simple numeric features.

In [ ]:
eu   = pd.read_excel(os.path.join(DATA_DIR, "Existing_Users.xlsx"))
ctrl = pd.read_excel(os.path.join(DATA_DIR, "Control_Group.xlsx"))
trt  = pd.read_excel(os.path.join(DATA_DIR, "Treatment_Group.xlsx"))

# Extract the poster's handle from each tweet URL  (x.com/{handle}/status/{id})
ctrl["tweet_handle"] = ctrl["URL"].str.extract(r"x\.com/([^/]+)/status", expand=False).str.strip().str.lower()
trt["tweet_handle"]  = trt["URL"].str.extract(r"x\.com/([^/]+)/status", expand=False).str.strip().str.lower()

tweet_handles = (
    set(ctrl["tweet_handle"].dropna()) | set(trt["tweet_handle"].dropna())
)
print(f"Unique tweet handles: {len(tweet_handles)}")

# Add handle to Existing_Users from their profile URL
eu["handle"] = eu["url"].str.extract(r"x\.com/([^/?]+)", expand=False).str.strip().str.lower()

# Filter to tweet authors; deduplicate by handle (keep the row with more followers if a
# handle somehow appears twice in EU — extremely rare, 3 cases found in diagnostics)
users = (
    eu[eu["handle"].isin(tweet_handles)]
    .sort_values("followers", ascending=False)
    .drop_duplicates(subset="handle")
    .copy()
    .reset_index(drop=True)
)
print(f"Matched in Existing_Users: {len(users)}")

# ── Compute simple derived features ───────────────────────────────────────────
users["creation_date"]   = pd.to_datetime(users["creation_date"],  errors="coerce")
users["detection_date"]  = pd.to_datetime(users["detection_date"], errors="coerce")
users["account_age_days"] = (
    (users["detection_date"] - users["creation_date"]).dt.days.clip(lower=0)
)
users["log_followers"] = np.log1p(users["followers"])
users["log_following"] = np.log1p(users["following"])
users["ff_ratio"]      = users["followers"] / (users["following"] + 1)
users["has_bio"]       = (
    users["description_short"].notna() &
    (users["description_short"].astype(str).str.strip() != "")
)
users["has_location"]  = users["location"].notna()

print("\nSimple feature summary:")
print(users[["followers","following","ff_ratio","account_age_days"]].describe().round(1))

---
## 2b. XPOZ Fallback for Unmatched Users

For tweet authors not found in `Existing_Users`, fetch basic profile data from XPOZ
(followers, following, bio, location, creation date, avatar) alongside the enriched fields —
so they enter the same feature pipeline rather than being dropped as NaN.

In [ ]:
# ── Identify handles missing from EU ──────────────────────────────────────────
all_tweet_handles = (
    set(ctrl["tweet_handle"].dropna()) | set(trt["tweet_handle"].dropna())
)
matched_handles   = set(users["handle"])
unmatched_handles = sorted(all_tweet_handles - matched_handles)
print(f"Total unique tweet handles  : {len(all_tweet_handles)}")
print(f"Matched in Existing_Users   : {len(matched_handles)}")
print(f"Unmatched (XPOZ fallback)   : {len(unmatched_handles)}")

# ── Debug: inspect field names returned by XPOZ for one handle ────────────────
xpoz = XpozClient(XPOZ_API_KEY)

if unmatched_handles:
    sample = xpoz.twitter.get_user(unmatched_handles[0], fields=[
        "followers_count", "following_count", "description", "created_at",
        "location", "profile_image_url", "name",
        "verified", "is_inauthentic_prob_score", "inauthentic_type",
        "avg_tweets_per_day_last_month",
    ])
    print(f"\nXPOZ response for '{unmatched_handles[0]}':")
    try:
        print(json.dumps(sample.__dict__, indent=2, default=str))
    except Exception:
        print(vars(sample))

In [ ]:
# ── Fetch all unmatched handles from XPOZ ─────────────────────────────────────
# Adjust field names below if the debug cell above showed different names.
XPOZ_FALLBACK_FIELDS = [
    # Basic profile — maps to EU columns
    "followers_count", "following_count", "description",
    "created_at", "location", "profile_image_url", "name",
    # Enriched — same as Layer 4
    "verified", "is_inauthentic_prob_score", "inauthentic_type",
    "avg_tweets_per_day_last_month",
]

XPOZ_FALLBACK_CACHE = os.path.join(OUT_DIR, "xpoz_fallback_cache.json")

if os.path.exists(XPOZ_FALLBACK_CACHE):
    with open(XPOZ_FALLBACK_CACHE, encoding="utf-8") as f:
        fallback_cache = json.load(f)
    print(f"Loaded fallback cache: {len(fallback_cache)} handles.")
else:
    fallback_cache = {}

fallback_errors = []
for i, handle in enumerate(unmatched_handles):
    if handle in fallback_cache:
        continue
    try:
        u = xpoz.twitter.get_user(handle, fields=XPOZ_FALLBACK_FIELDS)
        fallback_cache[handle] = {f: getattr(u, f, None) for f in XPOZ_FALLBACK_FIELDS}
        time.sleep(API_DELAY)
    except Exception as e:
        fallback_errors.append({"handle": handle, "error": str(e)})
        fallback_cache[handle] = {f: None for f in XPOZ_FALLBACK_FIELDS}

    if (i + 1) % 20 == 0:
        with open(XPOZ_FALLBACK_CACHE, "w", encoding="utf-8") as f:
            json.dump(fallback_cache, f, indent=2, default=str)
        print(f"  {i+1}/{len(unmatched_handles)} fetched ...")

with open(XPOZ_FALLBACK_CACHE, "w", encoding="utf-8") as f:
    json.dump(fallback_cache, f, indent=2, default=str)

print(f"\nDone. {len(fallback_errors)} errors.")
if fallback_errors:
    print("Errors:", fallback_errors[:5])

In [ ]:
# ── Build fallback rows and append to `users` ──────────────────────────────────
FIELD_MAP = {
    "followers_count":   "followers",
    "following_count":   "following",
    "description":       "description_short",
    "created_at":        "creation_date",
    "location":          "location",
    "profile_image_url": "avatar",
    "name":              "user_name",
}


def parse_created_at(v):
    """Handle both Unix timestamp strings ('1764640407') and ISO date strings ('2021-03-15T...')."""
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return pd.NaT
    try:
        return pd.Timestamp(int(v), unit="s")
    except (ValueError, TypeError):
        return pd.to_datetime(str(v), errors="coerce", utc=True).tz_localize(None)


# Initialize xpoz_cache here (cell 3 may not have run yet); cell 3 will load from file if it exists
if os.path.exists(XPOZ_CACHE):
    with open(XPOZ_CACHE, encoding="utf-8") as f:
        xpoz_cache = json.load(f)
else:
    xpoz_cache = {}

fallback_rows = []
for handle in unmatched_handles:
    rec = fallback_cache.get(handle, {})
    if all(v is None for v in rec.values()):
        continue

    row = {"handle": handle}
    for xpoz_field, eu_col in FIELD_MAP.items():
        row[eu_col] = rec.get(xpoz_field)

    fallback_rows.append(row)

    # Pre-seed xpoz_cache so the Layer 4 cell skips re-fetching these handles
    if handle not in xpoz_cache:
        xpoz_cache[handle] = {
            "verified":                      rec.get("verified"),
            "is_inauthentic_prob_score":     rec.get("is_inauthentic_prob_score"),
            "inauthentic_type":              rec.get("inauthentic_type"),
            "avg_tweets_per_day_last_month": rec.get("avg_tweets_per_day_last_month"),
        }

# Save so cell 3 loads the pre-seeded entries and skips these handles
with open(XPOZ_CACHE, "w", encoding="utf-8") as f:
    json.dump(xpoz_cache, f, indent=2, default=str)

if fallback_rows:
    fallback_df = pd.DataFrame(fallback_rows)

    fallback_df["creation_date"] = fallback_df["creation_date"].apply(parse_created_at)
    fallback_df["detection_date"] = pd.NaT
    fallback_df["account_age_days"] = (
        (pd.Timestamp.now() - fallback_df["creation_date"]).dt.days.clip(lower=0)
    )
    fallback_df["followers"]     = pd.to_numeric(fallback_df["followers"], errors="coerce")
    fallback_df["following"]     = pd.to_numeric(fallback_df["following"], errors="coerce")
    fallback_df["log_followers"] = np.log1p(fallback_df["followers"])
    fallback_df["log_following"] = np.log1p(fallback_df["following"])
    fallback_df["ff_ratio"]      = fallback_df["followers"] / (fallback_df["following"] + 1)
    fallback_df["has_bio"]       = (
        fallback_df["description_short"].notna() &
        (fallback_df["description_short"].astype(str).str.strip() != "")
    )
    fallback_df["has_location"]  = fallback_df["location"].notna()

    users = pd.concat([users, fallback_df], ignore_index=True)
    print(f"Appended {len(fallback_rows)} fallback rows.")
    print(f"  account_age_days non-null: {fallback_df['account_age_days'].notna().sum()}/{len(fallback_df)}")
else:
    print("No fallback rows to append.")

print(f"Total users after fallback: {len(users)}")

---
## 3. XPOZ SDK Enrichment (Layer 4)

Fetches 4 fields not available in `Existing_Users`:
- `verified` — verification status (key modulator of perceived credibility)
- `is_inauthentic_prob_score` — bot probability (0–1)
- `inauthentic_type` — e.g. `"BOT-LIKE ACTIVITY"` or `None`
- `avg_tweets_per_day_last_month` — recent posting activity

Results cached in `xpoz_cache.json`. Run the debug cell first to verify field names.

In [ ]:
# ── Debug: inspect one user to confirm exact attribute names from SDK ──────────
xpoz = XpozClient(XPOZ_API_KEY)
sample_handle = users["handle"].iloc[10]
print(f"Testing with handle: {sample_handle}")
test_user = xpoz.twitter.get_user(
    sample_handle,
    fields=["verified", "is_inauthentic_prob_score", "inauthentic_type",
            "avg_tweets_per_day_last_month"]
)
print("Sample user attributes:")
try:
    print(json.dumps(test_user.__dict__, indent=2, default=str))
except Exception:
    print(vars(test_user))

In [ ]:
# ── Full XPOZ collection run ───────────────────────────────────────────────────
# Cache is keyed by Twitter handle (not display name) to avoid ambiguity.
XPOZ_FIELDS = [
    "verified",
    "is_inauthentic_prob_score",
    "inauthentic_type",
    "avg_tweets_per_day_last_month",
]

if os.path.exists(XPOZ_CACHE):
    with open(XPOZ_CACHE, encoding="utf-8") as f:
        xpoz_cache = json.load(f)
    print(f"Loaded cache: {len(xpoz_cache)} users already fetched.")
else:
    xpoz_cache = {}

# Exclude already-cached handles so progress reflects only what's left to fetch
all_handles      = users["handle"].tolist()
handles_to_fetch = [h for h in all_handles if h not in xpoz_cache]
print(f"To fetch: {len(handles_to_fetch)} / {len(all_handles)} total")

xpoz_errors = []

for i, handle in enumerate(handles_to_fetch):
    try:
        u = xpoz.twitter.get_user(handle, fields=XPOZ_FIELDS)
        record = {field: getattr(u, field, None) for field in XPOZ_FIELDS}
        xpoz_cache[handle] = record
        time.sleep(API_DELAY)
    except Exception as e:
        xpoz_errors.append({"handle": handle, "error": str(e)})
        xpoz_cache[handle] = {f: None for f in XPOZ_FIELDS}

    if (i + 1) % 20 == 0:
        with open(XPOZ_CACHE, "w", encoding="utf-8") as f:
            json.dump(xpoz_cache, f, indent=2, default=str)
        print(f"  {i+1}/{len(handles_to_fetch)} fetched ...")

with open(XPOZ_CACHE, "w", encoding="utf-8") as f:
    json.dump(xpoz_cache, f, indent=2, default=str)

print(f"\nDone. {len(xpoz_errors)} errors.")
if xpoz_errors:
    print("First errors:", xpoz_errors[:3])

In [ ]:
# ── Parse XPOZ cache into dataframe columns ────────────────────────────────────
xpoz_rows = []
for handle in users["handle"]:
    rec = xpoz_cache.get(handle, {})
    xpoz_rows.append({
        "handle":                handle,
        "xpoz_verified":         rec.get("verified"),
        "xpoz_inauthentic_prob": rec.get("is_inauthentic_prob_score"),
        "xpoz_inauthentic_type": rec.get("inauthentic_type"),
        "xpoz_avg_tweets_day":   rec.get("avg_tweets_per_day_last_month"),
    })

xpoz_df = pd.DataFrame(xpoz_rows)
print("XPOZ features:")
print(xpoz_df.describe(include="all").T[["count","unique","top","mean"]].fillna(""))

---
## 4. Gemini Bio Scoring (Layer 2)

Scores each user's bio on:
- **Stance explicitness (1–5)**: how overtly pro-Russian is this bio?
  - 1 = no political signal (personal/hobby/blank)
  - 2 = vague nationalist or anti-Western tone
  - 3 = moderate political lean (anti-NATO, anti-Ukraine framing)
  - 4 = clearly pro-Russian ("Z", patriotic phrases, Russian flag emoji)
  - 5 = explicitly aggressive pro-Russian (Cyrillic slogans, direct support statements)
- **Account type** (categorical): `individual` / `journalist_or_media` / `political_figure` / `organization` / `anonymous_or_no_info`

In [ ]:
gemini = genai.Client(api_key=GEMINI_API_KEY)

BIO_SYSTEM = (
    "You are an expert analyst of social media accounts. "
    "Respond ONLY with a raw JSON object — no markdown, no code fences."
)

BIO_PROMPT = """Analyze this Twitter/X user bio and return scores.

User bio: {bio}

Score on two dimensions:

stance_explicitness (1-5): How overtly and explicitly pro-Russian is this bio?
  1 = No political signal at all (personal hobbies, blank, unrelated)
  2 = Vague nationalist or anti-Western tone, nothing explicit
  3 = Moderate political lean visible (anti-NATO, anti-Ukraine framing, general Russian patriotism)
  4 = Clearly pro-Russian (Z symbol, Russian flag emoji, phrases like "I support Russia", explicit anti-Ukraine)
  5 = Aggressively and explicitly pro-Russian (Cyrillic patriotic slogans, "Glory to Russia", direct military support)

account_type: Best fitting category for this account.
  Options: individual, journalist_or_media, political_figure, organization, anonymous_or_no_info

Respond with ONLY this JSON:
{{"stance_explicitness": <1-5>, "account_type": "<category>", "reasoning": "<one sentence>"}}"""


def _extract_json(raw):
    if not raw or not raw.strip():
        raise ValueError("Empty response")
    for attempt in [
        lambda s: json.loads(s),
        lambda s: json.loads(re.sub(r"^```(?:json)?\s*", "", s.strip(), flags=re.I).rstrip("```").strip()),
        lambda s: json.loads(re.search(r"\{[^{}]*\}", s, re.DOTALL).group()),
    ]:
        try:
            return attempt(raw)
        except Exception:
            pass
    raise ValueError(f"Cannot parse: {repr(raw[:200])}")


def score_bio(bio_text):
    prompt = BIO_PROMPT.format(bio=bio_text if bio_text and str(bio_text).strip() else "(no bio)")
    response = gemini.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=BIO_SYSTEM,
            temperature=0.1,
            max_output_tokens=512,
            thinking_config=types.ThinkingConfig(thinking_budget=0),
        ),
    )
    return _extract_json(response.text)


print("Gemini client ready. Test:")
print(score_bio("Proud Russian patriot. Z 🇷🇺"))

In [ ]:
# ── Run bio scoring ────────────────────────────────────────────────────────────
if os.path.exists(BIO_CACHE):
    with open(BIO_CACHE, encoding="utf-8") as f:
        bio_cache = json.load(f)
    print(f"Loaded bio cache: {len(bio_cache)} users.")
else:
    bio_cache = {}

bio_errors = []

for i, row in users.iterrows():
    handle = row["handle"]
    if handle in bio_cache:
        continue
    try:
        result = score_bio(row["description_short"])
        bio_cache[handle] = result
        time.sleep(API_DELAY)
    except Exception as e:
        bio_errors.append({"handle": handle, "error": str(e)})
        bio_cache[handle] = {"stance_explicitness": None, "account_type": None, "reasoning": None}

    if (i + 1) % 50 == 0:
        with open(BIO_CACHE, "w", encoding="utf-8") as f:
            json.dump(bio_cache, f, indent=2, ensure_ascii=False)
        print(f"  {i+1}/{len(users)} bios scored ...")

with open(BIO_CACHE, "w", encoding="utf-8") as f:
    json.dump(bio_cache, f, indent=2, ensure_ascii=False)

print(f"\nDone. {len(bio_errors)} errors.")

---
## 5. Gemini Avatar Vision (Layer 3)

Downloads each avatar image and asks Gemini to classify:
- **`avatar_type`**: `real_person_photo` / `flag_or_national_symbol` / `organization_logo` / `cartoon_or_illustration` / `default_or_empty`
- **`has_national_symbol`** (bool): Russian flag, Z symbol, coat of arms, military imagery, etc.

**Rationale for CN suppression:** Accounts with real-face avatars attract audiences more likely to engage with replies (CNs may be more effective). Flag/symbol avatars signal strong ideology and pre-filtered audiences (CNs may have less impact).

In [ ]:
AVATAR_SYSTEM = (
    "You are classifying Twitter/X profile avatars for a research study on disinformation. "
    "Respond ONLY with a raw JSON object — no markdown, no code fences."
)

AVATAR_PROMPT = """Classify this Twitter/X profile picture.

avatar_type: Choose the single best fitting category:
  real_person_photo     — clearly a photo of a real human face
  flag_or_national_symbol — flag, coat of arms, military symbol, or national emblem
  organization_logo     — brand, media outlet, or org logo
  cartoon_or_illustration — drawing, anime, meme, or non-photo graphic
  default_or_empty      — Twitter default egg, generic silhouette, blank, or landscape with no person

has_national_symbol (true/false): Does the image contain any national or political symbol?
  (e.g. Russian flag, Ukrainian flag, Z symbol, St. George ribbon, military insignia, any country flag)

Respond with ONLY this JSON:
{"avatar_type": "<category>", "has_national_symbol": <true/false>, "reasoning": "<one sentence>"}"""

HEADERS_LIST = [
    {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"},
    {"User-Agent": "Twitterbot/1.0"},
    {"User-Agent": "curl/7.68.0"},
]

def url_variants(url):
    """Generate size-variant URLs to try when the original fails."""
    variants = [url]
    for old, new in [
        ("_normal.", "_400x400."), ("_400x400.", "_normal."),
        ("_bigger.", "_400x400."), ("_mini.", "_normal."),
    ]:
        if old in url:
            variants.append(url.replace(old, new))
    # Also try stripping the size suffix entirely (some CDNs serve original)
    import re
    stripped = re.sub(r"_(normal|bigger|mini|400x400|200x200|orig)\.", ".", url)
    if stripped != url:
        variants.append(stripped)
    return list(dict.fromkeys(variants))  # deduplicate, preserve order


def detect_mime(image_bytes):
    if image_bytes[:4] == b"\x89PNG":
        return "image/png"
    if image_bytes[:4] == b"RIFF" and image_bytes[8:12] == b"WEBP":
        return "image/webp"
    if image_bytes[:6] in (b"GIF87a", b"GIF89a"):
        return "image/gif"
    return "image/jpeg"


def download_avatar(url, cache_path):
    """Try the URL and its size variants with multiple User-Agents. Returns bytes or None."""
    if os.path.exists(cache_path):
        data = open(cache_path, "rb").read()
        # Re-download if cached file is suspiciously small (likely an error page)
        if len(data) > 500:
            return data
        os.remove(cache_path)

    for try_url in url_variants(url):
        for headers in HEADERS_LIST:
            try:
                resp = requests.get(try_url, timeout=12, headers=headers)
                if resp.status_code == 200 and len(resp.content) > 500:
                    with open(cache_path, "wb") as f:
                        f.write(resp.content)
                    return resp.content
            except Exception:
                continue
    return None


def score_avatar(image_bytes):
    mime = detect_mime(image_bytes)
    response = gemini.models.generate_content(
        model=GEMINI_MODEL,
        contents=[
            types.Part.from_bytes(data=image_bytes, mime_type=mime),
            types.Part.from_text(text=AVATAR_PROMPT),
        ],
        config=types.GenerateContentConfig(
            system_instruction=AVATAR_SYSTEM,
            temperature=0.1,
            max_output_tokens=512,
            thinking_config=types.ThinkingConfig(thinking_budget=0),
        ),
    )
    return _extract_json(response.text)


print("Avatar scoring functions ready.")

In [ ]:
# ── Run avatar scoring ─────────────────────────────────────────────────────────
if os.path.exists(AVATAR_SCORE_CACHE):
    with open(AVATAR_SCORE_CACHE, encoding="utf-8") as f:
        avatar_cache = json.load(f)
    print(f"Loaded avatar cache: {len(avatar_cache)} users.")
else:
    avatar_cache = {}

# Clear entries that failed so they get retried with the improved downloader
avatar_cache = {
    k: v for k, v in avatar_cache.items()
    if v.get("avatar_type") is not None
    and v.get("reasoning") not in ("download failed",)
}
print(f"Kept {len(avatar_cache)} valid entries, cleared the rest for retry")

avatar_errors = []

for i, row in users.iterrows():
    handle = row["handle"]
    if handle in avatar_cache:
        continue

    url = str(row["avatar"]) if pd.notna(row["avatar"]) else ""
    if not url:
        avatar_cache[handle] = {"avatar_type": "default_or_empty", "has_national_symbol": False, "reasoning": "no url"}
        continue

    # Build a safe local filename from a hash of the URL
    ext = url.split(".")[-1].split("?")[0][:4]  # jpg / png / webp
    fname = hashlib.md5(url.encode()).hexdigest()[:12] + "." + ext
    cache_path = os.path.join(AVATAR_DIR, fname)

    image_bytes = download_avatar(url, cache_path)
    if image_bytes is None:
        avatar_cache[handle] = {"avatar_type": "default_or_empty", "has_national_symbol": False, "reasoning": "download failed"}
        continue

    try:
        result = score_avatar(image_bytes)
        avatar_cache[handle] = result
        time.sleep(API_DELAY)
    except Exception as e:
        avatar_errors.append({"handle": handle, "error": str(e)})
        avatar_cache[handle] = {"avatar_type": None, "has_national_symbol": None, "reasoning": None}

    if (i + 1) % 20 == 0:
        with open(AVATAR_SCORE_CACHE, "w", encoding="utf-8") as f:
            json.dump(avatar_cache, f, indent=2, ensure_ascii=False)
        print(f"  {i+1}/{len(users)} avatars scored ...")

with open(AVATAR_SCORE_CACHE, "w", encoding="utf-8") as f:
    json.dump(avatar_cache, f, indent=2, ensure_ascii=False)

print(f"\nDone. {len(avatar_errors)} errors.")

---
## 6. Build & Save Unified Dataset

In [ ]:
# ── Parse bio cache ────────────────────────────────────────────────────────────
bio_df = pd.DataFrame([
    {
        "handle":           h,
        "bio_stance_score": (bio_cache.get(h) or {}).get("stance_explicitness"),
        "bio_account_type": (bio_cache.get(h) or {}).get("account_type"),
    }
    for h in users["handle"]
])

# ── Parse avatar cache ─────────────────────────────────────────────────────────
avatar_df = pd.DataFrame([
    {
        "handle":                h,
        "avatar_type":           (avatar_cache.get(h) or {}).get("avatar_type"),
        "avatar_national_symbol": (avatar_cache.get(h) or {}).get("has_national_symbol"),
    }
    for h in users["handle"]
])

# ── Merge all layers on handle ─────────────────────────────────────────────────
feature_df = (
    users
    .merge(xpoz_df,   on="handle", how="left")
    .merge(bio_df,    on="handle", how="left")
    .merge(avatar_df, on="handle", how="left")
)

# Keep raw bio + avatar URL as reference columns (not used as analysis variables)
out_cols = [
    # identity — handle is the unambiguous join key; user_name kept as human-readable label
    "handle", "user_name",
    # raw (reference only)
    "description_short", "avatar",
    # Layer 1 — simple
    "followers", "following", "log_followers", "log_following",
    "ff_ratio", "account_age_days", "has_bio", "has_location",
    # Layer 2 — Gemini bio
    "bio_stance_score", "bio_account_type",
    # Layer 3 — Gemini avatar
    "avatar_type", "avatar_national_symbol",
    # Layer 4 — XPOZ
    "xpoz_verified", "xpoz_inauthentic_prob", "xpoz_inauthentic_type", "xpoz_avg_tweets_day",
]
feature_df = feature_df[[c for c in out_cols if c in feature_df.columns]]

out_path = os.path.join(OUT_DIR, "user_features.csv")
feature_df.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"Saved: {out_path}")
print(f"Shape: {feature_df.shape}")

In [ ]:
# ── Final summary ──────────────────────────────────────────────────────────────
print("=" * 60)
print("USER FEATURES DATASET SUMMARY")
print("=" * 60)
print(f"\nRows (users): {len(feature_df)}")
print(f"Columns     : {feature_df.shape[1]}")

print("\n--- Nulls per feature column ---")
analysis_cols = [c for c in feature_df.columns if c not in ["user_name","description_short","avatar"]]
nulls = feature_df[analysis_cols].isnull().sum()
print(nulls[nulls > 0].to_string())

print("\n--- Categorical distributions ---")
for col in ["bio_account_type", "avatar_type", "xpoz_inauthentic_type"]:
    if col in feature_df.columns:
        print(f"\n{col}:")
        print(feature_df[col].value_counts(dropna=False).to_string())

print("\n--- Verified status ---")
if "xpoz_verified" in feature_df.columns:
    print(feature_df["xpoz_verified"].value_counts(dropna=False).to_string())